# Coordinación semafórica — Onda verde en Av. Luis Elizondo (Distrito Tec)

**Reto Integrador – TC2008 (Sistemas Multiagente).**

Corredor de **3 intersecciones** sobre Av. Luis Elizondo:

1. **Fernando García Roel / Cantú Leal**
2. **Junco de la Vega** (cruce en T)
3. **Eugenio Garza Sada**

Este cuaderno calcula los **offsets** de coordinación, dibuja el **diagrama
espacio–tiempo** con bandas verdes y trayectorias, y evalúa el **ancho de banda**
y la **progresión vehicular** (qué tan bien se mantiene la onda verde).

> Nota: las distancias son aproximadas medidas en Google Maps y deben confirmarse.
> Los volúmenes son estimaciones razonables mientras se obtienen los datos reales.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Parámetros del corredor y justificación

**Distancias entre intersecciones** (medidas en Google Maps, eje del corredor):

| Tramo | Distancia |
|---|---|
| García Roel → Junco | 345 m |
| Junco → Garza Sada | 500 m |

**Velocidad de sincronía: 40 km/h.** Se justifica porque es una avenida urbana
dentro del Distrito Tec, con presencia de peatones, paradas y campus, donde una
progresión de 40 km/h es realista y segura (por debajo de arterias de 50–60 km/h).

**Ciclo semafórico:** C = 66 s, con **verde coordinado = 45 s**, amarillo = 3 s,
rojo = 18 s. Se da verde amplio a Av. Luis Elizondo por ser el corredor principal
(las transversales reciben el resto del ciclo).

In [ ]:
# --- Datos del corredor ---
intersections = pd.DataFrame({
    'id':     ['I1', 'I2', 'I3'],
    'nombre': ['Fernando García Roel', 'Junco de la Vega', 'Eugenio Garza Sada'],
    'dist_m': [0, 345, 845],   # acumulada sobre el corredor
})

C       = 66.0          # ciclo común (s)
verde   = 45.0          # verde coordinado (s)
amarillo = 3.0
rojo    = 18.0
v_kmh   = 40.0          # velocidad de sincronía
v       = v_kmh * 1000 / 3600   # m/s

print(f'Velocidad de sincronía: {v_kmh} km/h = {v:.2f} m/s')
print(f'Ciclo C = {C}s  (verde {verde}s / amarillo {amarillo}s / rojo {rojo}s)')
print(f'Split de verde coordinado = {verde/C:.0%}')
intersections

## 2. Cálculo de offsets

El **offset** es el desfase de arranque del verde entre intersecciones para que
un pelotón viaje sin detenerse:

$$ t_i = \frac{d_i - d_{i-1}}{v} \qquad Offset_i = (Offset_{i-1} + t_i)\bmod C $$

Para el sentido contrario el cálculo se hace desde la última intersección.

In [ ]:
def offsets_forward(d, v, C):
    o = [0.0]
    for i in range(1, len(d)):
        o.append((o[-1] + (d[i] - d[i-1]) / v) % C)
    return o

def offsets_reverse(d, v, C):
    o = [0.0] * len(d)
    for i in range(len(d) - 2, -1, -1):
        o[i] = (o[i+1] + (d[i+1] - d[i]) / v) % C
    return o

d = intersections['dist_m'].values
intersections['offset_fwd'] = np.round(offsets_forward(d, v, C), 2)
intersections['offset_rev'] = np.round(offsets_reverse(d, v, C), 2)
intersections

Los offsets (0 / 31.1 / 10.1 s en sentido García Roel→Garza Sada) coinciden con el
plan de coordinación del corredor: cada cruce abre su verde justo cuando el pelotón
que viaja a 40 km/h llega a él.

## 3. Diagrama espacio–tiempo (entregable clave)

- Eje X = tiempo, eje Y = posición sobre el corredor.
- Las **barras verdes** son las ventanas de verde coordinado de cada intersección.
- Las **líneas** son trayectorias de vehículos (pendiente = velocidad de sincronía).
- Si una trayectoria cruza todas las barras verdes, la **onda verde se mantiene**.

In [ ]:
def diagrama_espacio_tiempo(inter, C, g, v, n_ciclos=3, sentido='fwd'):
    fig, ax = plt.subplots(figsize=(13, 7))
    dmax = inter['dist_m'].max()
    col = 'offset_fwd' if sentido == 'fwd' else 'offset_rev'

    # Bandas verdes por intersección
    for _, row in inter.iterrows():
        y = row['dist_m']
        for k in range(n_ciclos):
            start = row[col] + k * C
            ax.hlines(y, start, start + g, color='green', linewidth=10, alpha=0.45)
        ax.text(-6, y, row['id'], va='center', ha='right', fontweight='bold')

    # Trayectorias del pelotón
    for k in range(n_ciclos + 1):
        if sentido == 'fwd':
            t0 = inter['offset_fwd'].iloc[0] + k * C
            ax.plot([t0, t0 + dmax / v], [0, dmax], color='blue', lw=2.2,
                    label='Pelotón (onda verde)' if k == 0 else None)
        else:
            t0 = inter['offset_rev'].iloc[-1] + k * C
            ax.plot([t0, t0 + dmax / v], [dmax, 0], color='red', lw=2.2,
                    label='Pelotón (onda verde)' if k == 0 else None)

    ax.set_title(f'Diagrama espacio-tiempo — sentido {"García Roel→Garza Sada" if sentido=="fwd" else "Garza Sada→García Roel"}')
    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel('Distancia sobre el corredor (m)')
    ax.set_xlim(0, C * n_ciclos)
    ax.set_ylim(-40, dmax + 60)
    ax.grid(True, linestyle='--', alpha=0.4)
    ax.legend(loc='upper right')
    plt.tight_layout(); plt.show()

diagrama_espacio_tiempo(intersections, C, verde, v, sentido='fwd')
diagrama_espacio_tiempo(intersections, C, verde, v, sentido='rev')

## 4. Ancho de banda y progresión

El **ancho de banda** es el rango de tiempo (s) dentro del cual un vehículo puede
entrar al corredor y encontrar verde en **todas** las intersecciones. Es la medida
de qué tan buena es la coordinación.

In [ ]:
def ancho_de_banda(offsets, d, v, C, g, n=4000):
    ts = np.linspace(0, C, n, endpoint=False)
    ok = np.ones(n, dtype=bool)
    for off, di in zip(offsets, d):
        arr = (ts + di / v) % C            # instante de llegada a la intersección
        ok &= ((arr - off) % C) <= g       # ¿cae en su ventana verde?
    # corrida contigua más larga (circular)
    run = best = 0
    for b in np.concatenate([ok, ok]):
        run = run + 1 if b else 0
        best = max(best, run)
    return min(best, n) / n * C

of = offsets_forward(d, v, C)
orv = offsets_reverse(d, v, C)
bw_f = ancho_de_banda(of, d, v, C, verde)
bw_r = ancho_de_banda(orv, d, v, C, verde)

print(f'Ancho de banda  García Roel→Garza Sada: {bw_f:5.1f} s  ({bw_f/C:.0%} del ciclo)')
print(f'Ancho de banda  Garza Sada→García Roel: {bw_r:5.1f} s  ({bw_r/C:.0%} del ciclo)')
print(f'Eficiencia bidireccional promedio:      {(bw_f+bw_r)/2/C:.0%}')

## 5. Evaluación de la progresión (¿llega en verde?)

Se simula un vehículo que sale en el instante óptimo y se verifica si llega en
verde a cada intersección.

In [ ]:
def evalua_progresion(inter, v, C, g, sentido='fwd'):
    d = inter['dist_m'].values
    nombres = inter['nombre'].values
    n = len(d)
    off = offsets_forward(d, v, C) if sentido == 'fwd' else offsets_reverse(d, v, C)
    idxs = range(n) if sentido == 'fwd' else range(n - 1, -1, -1)
    base = d[0] if sentido == 'fwd' else d[-1]
    t0 = off[0] if sentido == 'fwd' else off[-1]
    out = []
    for i in idxs:
        tt = abs(d[i] - base) / v
        arr = (t0 + tt) % C
        en_verde = ((arr - off[i]) % C) <= g + 1e-9   # tolerancia numérica
        out.append({'interseccion': nombres[i], 't_llegada_s': round(tt, 1),
                    'offset_s': round(off[i], 2), 'llega_en_verde': en_verde})
    return pd.DataFrame(out)

ev_f = evalua_progresion(intersections, v, C, verde, 'fwd')
ev_r = evalua_progresion(intersections, v, C, verde, 'rev')
print('Sentido García Roel -> Garza Sada')
print(ev_f.to_string(index=False))
print('\nSentido Garza Sada -> García Roel')
print(ev_r.to_string(index=False))
print(f'\nProgresión en verde  fwd: {ev_f.llega_en_verde.mean():.0%} | rev: {ev_r.llega_en_verde.mean():.0%}')

## 6. Escenarios de demanda (mañana / mediodía / tarde)

Mientras se obtienen los **datos reales** de volumen, se usan estimaciones (veh/h
por acceso). La coordinación (offsets) no cambia, pero la demanda afecta colas y
si la onda verde se **rompe** por saturación.

> Sustituir estos valores por los datos reales cuando estén disponibles.

In [ ]:
demanda = pd.DataFrame({
    'periodo':   ['Mañana (7-9)', 'Mediodía (13-15)', 'Tarde (18-20)'],
    'Elizondo_vph': [900, 650, 1100],   # estimado, corredor principal
    'transversal_vph': [500, 350, 600], # estimado, suma de transversales
})
# Grado de saturación aproximado (capacidad ~ verde/ciclo * 1800 vph por carril)
cap_eliz = (verde / C) * 1800 * 3   # 3 carriles
demanda['x_Elizondo'] = (demanda['Elizondo_vph'] / cap_eliz).round(2)
demanda['onda_estable'] = demanda['x_Elizondo'] < 0.85
print(f'Capacidad estimada Elizondo: {cap_eliz:.0f} vph')
demanda

## 7. Conclusiones preliminares (preguntas clave del reto)

- **¿Es posible coordinar el corredor?** Sí: con v=40 km/h y C=66 s, los offsets
  (0 / 31.1 / 10.1 s) logran una banda verde de ~45 s (68% del ciclo) en el sentido
  principal.
- **¿Qué tan estable es la progresión?** Excelente en el sentido coordinado (100%
  en verde); en el sentido contrario la banda baja (~21 s), típico en corredores
  asimétricos.
- **¿Qué pasa al aumentar la demanda?** En la tarde el grado de saturación sube;
  si supera ~0.85 la onda verde empieza a romperse por colas residuales.
- **¿Sensibilidad a la velocidad?** Cambiar v desplaza los offsets; conviene probar
  35–45 km/h y observar el ancho de banda (repetir el cálculo cambiando `v_kmh`).
- **Limitaciones urbanas:** cruce en T de Junco, vueltas e incorporaciones y
  peatones reducen la capacidad efectiva y el ancho de banda real.

_Estos resultados se complementan con la simulación multiagente en Unity (carros y
semáforos como agentes) y sus métricas de cola y tiempo de espera._